# Chapter 09: Sixteen minutes

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "09"
__file__ = os.path.abspath("run.py")

## The scene

Raj Malhotra's statement is two paragraphs long and very calm. He left the administration building
at about 23:30, walked straight to his car, and drove home. Varma puts the badge log and the gate
camera log next to it. "Calm is not the same as true," she says. "Walk me through it, minute by
minute. Then make LANTERN do the same, and show me where it slips."

The evidence: [`case/statements/malhotra.md`](../../case/statements/malhotra.md),
[`case/badge-log.csv`](../../case/badge-log.csv) and
[`case/site-facts.md`](../../case/site-facts.md).

## The concept

A model produces one token at a time, and every token it has written becomes input for the next.
When you ask for the answer straight away, the model must jump from the evidence to the verdict in
a single token: "yes" or "no". When you ask it to write the steps first, each step becomes part of
the context the final answer is generated from.

That is all chain of thought is: **spending output tokens on intermediate results so the final
answer is conditioned on them.** It comes in three strengths.

| Style | Prompt | Who chooses the steps |
|---|---|---|
| Direct | "Answer yes or no." | Nobody. One jump. |
| Free CoT | "Think step by step." | The model. |
| Set steps | "1. Quote X. 2. Quote Y. 3. Subtract..." | You. |

```mermaid
flowchart LR
    E[Evidence + question] --> D[Direct:<br/>verdict token]
    E --> F[Free CoT:<br/>model invents steps] --> FV[verdict]
    E --> S1[Step 1: quote] --> S2[Step 2: quote] --> S3[Step 3: subtract] --> S4[...] --> SV[verdict]
    S1 -.->|each step is context<br/>for the next| S2
```

## How it works

![Direct, free and set-step reasoning](../../docs/diagrams/09-cot-paths.svg)

The script asks the same question four ways, with the same evidence (Malhotra's three badge rows
and the walking times):

1. **Direct**: answer yes or no.
2. **Step by step**: the classic "think step by step" suffix.
3. **Set steps**: eight numbered steps you wrote, each small enough to check.
4. **Model reads, program reasons**: keep the two timestamps the model quoted in step 3, and do
   the arithmetic in Python.

```mermaid
sequenceDiagram
    participant You
    participant LANTERN
    participant Python
    You->>LANTERN: Step 1: copy the ADMIN-MAIN time
    LANTERN-->>You: 23:31:02
    You->>LANTERN: Step 2: copy the first GATE-2-PED time
    LANTERN-->>You: 23:47:09
    You->>Python: subtract, compare with walking times
    Python-->>You: 16.1 min elapsed, 11.1 unaccounted, stairwell route (7 min) fits
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import re

from lantern import case, chat, learn
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

chapter("09", "Sixteen minutes", """
Raj Malhotra's statement is two paragraphs long and very calm. He left the administration
building at about 23:30, walked straight to his car, and drove home. Varma puts the badge log and
the gate camera log next to it. "Calm is not the same as true," she says. "Walk me through it,
minute by minute. Then make LANTERN do the same, and show me where it slips."
""")

badge = case.read("badge-log.csv")
rows = "\n".join(line for line in badge.splitlines()
                 if "Malhotra" in line or line.startswith("time"))
walks = "\n".join(case.read("site-facts.md").splitlines()[2:5])
evidence("badge-log.csv (R. Malhotra rows)", rows)
evidence("site-facts.md (walking times)", walks)

# Every prompt gets the same material: Malhotra's badge rows and the walking times.
material = f"""<badge_log rows="R. Malhotra only">
{rows}
</badge_log>
<walking_times>
{walks}
</walking_times>"""
QUESTION = ("Malhotra says he left the administration building at about 23:30 and walked "
            "straight to his car at Gate 2. Do the records support that?")

PROMPTS = {
    "direct": f"{material}\n\n{QUESTION} Answer yes or no, then one sentence.",
    "step by step": f"{material}\n\n{QUESTION} Think step by step, then give your verdict.",
    "set steps": f"""{material}

{QUESTION}
Work through these steps in order. Show each step on its own line.
1. Copy the first row of the badge log (the ADMIN-MAIN row). Write its time.
2. Copy the second row of the badge log (the first GATE-2-PED row). Write its time.
3. Minutes from step 1 to step 2 (round to whole minutes).
4. Walking time, administration building to Gate 2.
5. Unaccounted minutes = step 3 minus step 4.
6. Stairwell C route = (administration building to stairwell C top) + (stairwell C bottom to
   Gate 2). Write the sum.
7. Is the step 6 sum smaller than the step 3 minutes? Answer yes or no.
8. Verdict in one sentence: is "walked straight to his car" supported or not supported?""",
}

note("Each prompt is an entry in the `PROMPTS` dict in run.py. All share `material`.")


def verdict(text: str) -> int:
    """0 if the reply says the records support him, 1 if not supported, 2 if it never says."""
    low = text.lower()
    if re.search(r"\bnot\s+supported\b|\bdo(?:es)?\s+not\s+support\b|\bunsupported\b", low):
        return 1
    if re.search(r"\bsupported\b", low):
        return 0
    first = re.search(r"\b(yes|no)\b", low[:60])
    return {"yes": 0, "no": 1}.get(first.group(1), 2) if first else 2


def ending(text: str) -> str:
    """The end of a long reply, where the verdict is."""
    return text.strip()[-300:]


VERDICTS = ["the records support him", "not supported", "no clear verdict"]
p_direct = learn.predict(
    "The direct prompt must answer yes or no in one jump. What will it say about Malhotra's "
    "account?", VERDICTS)
p_tokens = None

results = {}
for name, prompt in PROMPTS.items():
    step(f"{len(results) + 1}. {name}")
    if name == "step by step":
        p_tokens = learn.predict_number(
            f"The direct answer used {results['direct'].output_tokens} output tokens. How many "
            "will \"think step by step\" use?", unit="tokens")
    if name == "set steps":
        p_steps = learn.predict(
            "You wrote the eight steps. Will the verdict at the end be right (not supported)?",
            VERDICTS)
    r = chat(prompt, temperature=0, max_tokens=1000)
    results[name] = r
    reply(f"LANTERN, {name}", r.text,
          meta=f"{r.prompt_tokens} in, {r.output_tokens} out, {r.seconds:.0f}s")
    if name == "direct":
        learn.reveal(p_direct, answer=verdict(r.text),
                     why="A one-word verdict is produced in a single jump from the evidence. The "
                     "16 minutes between the two badge times has to be noticed without any "
                     "written step to notice it in.")
    if name == "step by step":
        learn.reveal(p_tokens, actual=r.output_tokens, tolerance=0.5,
                     why="Free chain of thought spends output tokens on steps the model chooses "
                     "for itself, and a small model chooses many. Every one is paid for in time.")
    if name == "set steps":
        note("Read it line by line. Every step is a claim you can check against the evidence.")
        learn.reveal(p_steps, answer=verdict(ending(r.text)),
                     why="Set steps make each slip visible, but a right verdict can still sit on "
                     "a wrong step: read step 4 against the walking times. The written reasoning "
                     "is not always how the model reached its answer.")

### 4. The model reads, the program reasons

In [ ]:
step("4. The model reads, the program reasons")
note("Keep the steps the model got right (copying times), and give the arithmetic to code.")
times = re.findall(r"\b(\d\d:\d\d:\d\d)\b", results["set steps"].text)
from_model = len(times) >= 2
p_gap = learn.predict_number(
    "Python takes the two times LANTERN copied and subtracts the 5 minute walk. How many minutes "
    "will be unaccounted for?", unit="minutes")
if from_model:
    left, gate = times[:2]
else:
    note("LANTERN did not quote two times, so the program reads them from the badge log.")
    left, gate = rows.splitlines()[1][:8], rows.splitlines()[2][:8]


def minutes(t: str) -> float:
    h, m, s = (int(x) for x in t.split(":"))
    return h * 60 + m + s / 60


gap = minutes(gate) - minutes(left)
table = Table(title="Times quoted by LANTERN, arithmetic by Python")
table.add_column("step")
table.add_column("value", justify="right")
table.add_column("source")
table.add_row("Left admin", left, "LANTERN, step 1")
table.add_row("Passed Gate 2", gate, "LANTERN, step 2")
table.add_row("Elapsed", f"{gap:.1f} min", "Python")
table.add_row("Walk admin to Gate 2", "5 min", "site facts")
table.add_row("Unaccounted", f"{gap - 5:.1f} min", "Python")
table.add_row("Stairwell C route", "3 + 4 = 7 min", "site facts")
table.add_row("Stairwell route fits?", "yes" if 7 <= gap else "no", "Python")
console.print(table)
learn.reveal(p_gap, actual=round(gap - 5, 1), tolerance=0.1,
             why="Once the times are copied, the rest is arithmetic, and arithmetic is what a "
             "program never gets wrong. The model's job shrinks to reading, which it does well.")

cost = Table(title="What each style cost")
for col in ("style", "output tokens", "seconds"):
    cost.add_column(col, justify="right" if col != "style" else "left")
for name, r in results.items():
    cost.add_row(name, str(r.output_tokens), f"{r.seconds:.0f}")
console.print(cost)
note("Reasoning out loud costs tokens and time. It pays when each step can be checked.")

learn.record(
    direct_says_supported=verdict(results["direct"].text) == 0,
    cot_says_not_supported=verdict(ending(results["step by step"].text)) == 1,
    steps_says_not_supported=verdict(ending(results["set steps"].text)) == 1,
    direct_output_tokens=results["direct"].output_tokens,
    cot_output_tokens=results["step by step"].output_tokens,
    steps_output_tokens=results["set steps"].output_tokens,
    direct_seconds=round(results["direct"].seconds, 2),
    cot_seconds=round(results["step by step"].seconds, 2),
    steps_seconds=round(results["set steps"].seconds, 2),
    times_from_model=from_model,
    times_quoted_right=(left, gate) == ("23:31:02", "23:47:09"),
    elapsed_minutes=round(gap, 1),
    unaccounted_minutes=round(gap - 5, 1),
    stairwell_fits=7 <= gap,
)

your_turn(
    "09",
    check=[
        "In the set steps reply, check steps 1 and 2 against the badge-log panel, then step 4 "
        "against the walking times panel. Which step slips?",
        "Where did the direct and step by step replies get their minutes? Compare them with the "
        "table that Python filled in.",
        "In the cost table, how many times more output tokens did step by step use than direct?",
    ],
    exercises=[
        "In run.py, replace the `rows = ...` filter with `rows = badge`, and change "
        "<badge_log rows=\"R. Malhotra only\"> to <badge_log> in `material`. "
        "Which steps fail first?",
        "In run.py, in the \"set steps\" prompt, change step 4 to 'Copy the line from "
        "walking_times that mentions Gate 2 staff car park.' Does quoting fix it?",
        "No edit: run LANTERN_MODEL=qwen3:4b make lesson N=09. Which set steps does the bigger "
        "model get right?",
        "In run.py, in the PROMPTS loop, add think=(name == \"direct\") to the chat(...) call and "
        "raise max_tokens to 4000. Compare direct with step by step in the cost table.",
    ],
)

finding("""
Malhotra badged out of the administration building at 23:31 and out of Gate 2 at 23:47. Sixteen
minutes for a five minute walk. A route down stairwell C takes seven, which leaves nine minutes
inside the stairwell, and Petrov heard the bang at about 23:40.

There is more. The gate camera shows his car leaving at 23:47 and coming back at 23:59, a minute
after Petrov's call. Malhotra told DS Iyer he was home by 00:05. He was never more than twelve
minutes from Pier 9.
""")
save_shot("09-chain-of-thought")

## The nuances

**1. Direct answers hide the error.** On our run the direct prompt said "yes": the records support
Malhotra. It even quoted 23:31 and 23:47 correctly, then said the timing "aligns with his claim",
without noticing the 16 minutes between those times. There is nothing to audit in a one-word verdict.

**2. Free CoT is not a fix on its own.** "Think step by step" produced 890 tokens. It copied all
three badge rows correctly, then wrote "From 23:31:02 to 23:47:09 is 6 minutes", called that
consistent with the walk to Gate 2, and concluded "Yes, the records support Malhotra's claim". More
steps gave it more room to slip, and nothing in the prompt told it which step mattered. (An earlier
run reached "not supported" by a different wrong road.) Its reasons would not survive Marcus Hale.

**3. Set steps make every slip visible.** The eight-step prompt got steps 1 to 3 exactly right
(23:31:02, 23:47:09, 16 minutes), then wrote "16 minutes" for the walking time in step 4, where the
evidence says 5. Every later step inherited the mistake. The verdict was still "not supported". A
right answer with wrong steps is a warning, not a pass: the written reasoning is not necessarily
how the model reached its answer (it is not always **faithful**).

**4. Let a program do what programs do.** The model was reliable at reading (copying timestamps)
and unreliable at arithmetic and comparison. Step 4 keeps the reading and gives the rest to Python.
This is the strongest pattern for small models: the model extracts, code computes. It is also the
version you can defend in court.

**5. CoT costs tokens and time.** Direct: 67 output tokens, 2 seconds. Free CoT: 890 tokens,
10 seconds. Set steps: 215 tokens, 3 seconds. The seconds are wall clock per call, as the cost
table prints them: model load, prompt processing and generation. Set steps are cheaper than free
CoT because you decided what to write.

**6. CoT can hurt simple tasks.** For a lookup ("what time did he badge out?") reasoning adds
tokens and chances to drift. Use it when the answer depends on combining several facts.

**7. Is chain of thought still relevant?** Reasoning models (chapter 10) now generate their own
hidden chain of thought before answering, and for them "think step by step" adds little and can
even interfere. CoT prompting still matters when:

- the model is small or has no reasoning mode (like most models you can run on a laptop);
- you need the steps **in the answer** so a person can audit them;
- you want to force a specific procedure, such as "quote before you compute".

For a frontier reasoning model, give it the procedure as requirements, not as a script to recite.

## On a hosted model

```bash
make lesson N=09 TIER=hosted     # needs ANTHROPIC_API_KEY in .env and `uv sync --extra cloud`
make lesson N=09 TIER=hosted REPLAY=1   # once a run has been recorded, no key needed
```

The hosted path sends the four prompts to claude-haiku-5-5 as user turns. With `think: false` in
the defaults, `cloud.py` sends thinking disabled, so the only chain of thought is the one written
in the answer, as on the local tier. Claude does not take `temperature`, `top_k`, `top_p` or
`seed`; `cloud.py` leaves out `temperature=0` and reports it. The Python step that subtracts the
timestamps runs the same on both tiers, and no step is skipped.

| Measure | Local: qwen3:1.7b, Ollama | Hosted: claude-haiku-5-5 |
|---|---|---|
| Direct: verdict, output tokens | "yes", 67 | not run |
| Free CoT: verdict, output tokens | "yes", 890 (counted 16 minutes as 6) | not run |
| Set steps: verdict, output tokens | "not supported", 215 (step 4 wrote 16 instead of 5) | not run |
| Model reads, program reasons | 16.1 minutes elapsed, 11.1 unaccounted | not run |
| Free CoT tokens against direct | about 13 times | not run |

Local numbers are from the screenshot's run (qwen3:1.7b, local tier, Apple M1 Pro 16 GB,
23 November 2026).

What the hosted run will test: whether the wrong direct "yes", the miscounted minutes in free CoT
and the slip in step 4 are small-model effects. That free CoT costs many times the tokens of a direct answer, and that
letting code do the arithmetic gives the answer you can defend, should hold on both.

## Try this

1. Remove the `rows="R. Malhotra only"` filter and give all eleven badge rows. Which steps fail
   first?
2. Change step 4 to "Copy the line from walking_times that mentions Gate 2 staff car park." Does
   quoting fix it?
3. Run the set-steps prompt with `LANTERN_MODEL=qwen3:4b`. Which steps does the bigger model get
   right?
4. Ask the direct question with `think=True` (see chapter 10) and compare the token cost with free
   CoT.

## Master it

After this chapter you can explain chain of thought in terms of tokens, choose between a direct
answer, free reasoning and set steps, spot a right verdict resting on a wrong step, and write the
code half of a "model reads, program reasons" check.

**Predict, then see.** `make lesson N=09` stops four times to ask what will happen: what the
direct prompt says about Malhotra, how many tokens "think step by step" spends, whether your set
steps reach the right verdict, and how many minutes Python finds unaccounted for. Each answer is
checked against your own run, not a fixed key. Add `ARGS=--no-predict` to watch without stopping.

**Build it.** Write the alibi check in [`practice/exercise.py`](practice/exercise.py): badge
swipes against a camera log and a walking time, returning the gap and whether the alibi holds.

```bash
make exercise N=09          # 7 offline tests on the real case files, no model needed
make hint N=09              # one hint at a time
make solution N=09          # the reference, once you have tried
```

**Check yourself.** `make quiz N=09`: six questions, each with the reason behind the answer.

**Break it.** `make challenge N=09` checks your latest run against three targets: a direct answer
that notices the gap in 150 tokens or fewer; set steps in 120 tokens with the verdict and both
times still right; and a model that misreads a time, so Python does perfect arithmetic on the
wrong numbers.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 09

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 09

In [ ]:
%run ../../scripts/learn.py hint 09

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 09

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

Malhotra badged out of the administration building at 23:31 and out of Gate 2 at 23:47. Sixteen
minutes for a five minute walk. A route down stairwell C takes seven, which leaves nine minutes
inside the stairwell, and Petrov heard the bang at about 23:40.

There is more. The gate camera shows his car leaving at 23:47 and coming back at 23:59, a minute
after Petrov's call. Malhotra told DS Iyer he was home by 00:05. He was never more than twelve
minutes from Pier 9.

**Next:** [Chapter 10: The locked door](../10-thinking-mode/README.md)